# 02 — Exploratory Data Analysis (Phase 2)

**Goal:** look *inside* the names and addresses to learn **how the same business gets written differently**.
Every difference we find here becomes a candidate cleaning step (Phase 6) or a feature (Phase 8).

Questions we answer:
1. **Names:** capitalization, other writing scripts, punctuation, noise, website-style names, common words, legal suffixes.
2. **Addresses:** empty / placeholder values, abbreviations, state formats, numbers and postal codes, landmarks, word order.
3. **Country:** which countries exist, and do matching records ever have different countries?

Plain pandas only.

## Why we work on a sample

Counting words in 5 million names creates ~20 million word rows, which is too much for 8 GB of RAM.
For **percentages and "most common" lists**, a random sample of 200,000 rows per file is plenty:
a percentage measured on 200k rows is accurate to about ±0.2 percentage points.

Exact counts (rows, empty values, duplicates) were already computed on the **full** files in notebook 01.

`random_state=0` makes the sample the same every time we run the notebook (reproducible).

In [1]:
import gc

import pandas as pd

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 100)

TRAIN_DIR = "../dataset/train/"
TEST_DIR = "../dataset/test/"
SAMPLE_SIZE = 200_000

files = {
    ("train", "S1"): TRAIN_DIR + "train_source1.tsv",
    ("train", "S2"): TRAIN_DIR + "train_source2.tsv",
    ("train", "S3"): TRAIN_DIR + "train_source3.tsv",
    ("test", "S1"): TEST_DIR + "test_source1.tsv",
    ("test", "S2"): TEST_DIR + "test_source2.tsv",
    ("test", "S3"): TEST_DIR + "test_source3.tsv",
}

pieces = []
for (split, source), path in files.items():
    df = pd.read_csv(path, sep="\t", keep_default_na=False)
    piece = df.sample(SAMPLE_SIZE, random_state=0)
    piece.insert(0, "split", split)
    piece.insert(1, "source", source)
    pieces.append(piece)
    del df
    gc.collect()
    print("sampled", split, source)

sample = pd.concat(pieces, ignore_index=True)
del pieces
print(sample.shape)
sample.head()

sampled train S1


sampled train S2


sampled train S3


sampled test S1


sampled test S2


sampled test S3
(1200000, 6)


,split,source,entity_id,business_name,business_address,country
0,train,S1,S1-889749447,Jagsakti Foodworks Group,"Gata No 98, Nasirpur, Sambhal Road, Mainather, Sambhal, Moradabad, Uttar Pradesh",India
1,train,S1,S1-921626517,Traum Peak Logistics P.C.,"82 Rogers Street, Atlanta, GA",US
2,train,S1,S1-582122760,Vaibhav Formulations LLP,"15/64 Garden House Elankom Gardens, Vellayambalam, Thiruvananthapuram, Trivandrum, Kerala",India
3,train,S1,S1-441617599,"Gilly Brunell, DPM Inc.","28486 Lamar Avenue, Chisago City, MN",US
4,train,S1,S1-338791606,Investors Textiles Private Limited,"G-195, Ist Floor Sector-10 Dlf, Faridabad, Haryana",India


Because the table has `split`, `source` and `country` columns, we can compare groups with
`groupby(["source", "country"])`. Most tables below have one row per (source, country).

A small helper: `pct(mask)` turns a True/False column into "percent True per group".

In [2]:
def pct(mask):
    """Percent of rows where `mask` is True, per (split, source, country)."""
    return (mask.groupby([sample["split"], sample["source"], sample["country"]]).mean() * 100).round(2)

names = sample["business_name"]
addresses = sample["business_address"]

## ⚠ A pandas regex trap we must know about

pandas 3 stores text in a fast format (pyarrow). For `.str.replace`, `.str.contains` and `.str.match` it uses a
regex engine where **`\w` means only English letters, digits and `_`**. Accented letters (`é`) and Indian letters are
treated as *symbols*. Watch what happens to `"École Límited"`:

In [3]:
demo = pd.Series(["École Límited", "राम Ltd"])
print("with \\w      :", demo.str.replace("[^\\w\\s]", " ", regex=True).tolist())
print("with \\p{...} :", demo.str.replace("[^\\p{L}\\p{N}\\p{M}\\s]", " ", regex=True).tolist())

with \w      : [' cole L mited', '    Ltd']
with \p{...} : ['École Límited', 'राम Ltd']


The fix is to use Unicode categories instead of `\w`:

| Pattern | Means |
|---|---|
| `\p{L}` | any **letter**, in any alphabet (a, é, र, ர ...) |
| `\p{N}` | any **digit** |
| `\p{M}` | **marks** — the vowel signs that Indian scripts attach to letters (`ा`, `ि`, `்`) |

So "not a letter, digit, mark or space" = `[^\p{L}\p{N}\p{M}\s]`. We use this everywhere below,
and we will reuse it when we write our cleaning functions in Phase 6.

---
# Part A — Business names

## A1. Length

Median length differs by country (France ~19–20, US ~22–24, India ~27–28 characters, because Indian names often
carry `Private Limited`). Within a country, Source 2/3 have longer "tails" (bigger max) because noise gets added.

In [4]:
sample.assign(name_length=names.str.len()).groupby(["source", "country"])["name_length"].describe().round(1)

count  mean  std  min   25%   50%   75%   max
source country                                                  
S1     France    29823.0  19.5  5.3  6.0  16.0  19.0  23.0  45.0
       India    173596.0  26.4  7.6  5.0  21.0  27.0  32.0  88.0
       US       196581.0  22.5  7.5  3.0  17.0  22.0  27.0  63.0
S2     France    28695.0  21.1  7.6  2.0  16.0  20.0  26.0  55.0
       India    174767.0  27.9  9.0  2.0  21.0  28.0  34.0  93.0
       US       196538.0  23.8  8.6  2.0  18.0  23.0  29.0  74.0
S3     France    28548.0  21.1  8.1  2.0  16.0  20.0  26.0  61.0
       India    174881.0  27.4  9.7  2.0  21.0  27.0  34.0  97.0
       US       196571.0  24.2  9.2  2.0  18.0  24.0  30.0  80.0

## A2. Capitalization

`name == name.str.upper()` is True when a name has no lowercase letters.

In [5]:
pd.DataFrame({
    "ALL UPPER %": pct(names == names.str.upper()),
    "all lower %": pct(names == names.str.lower()),
})

ALL UPPER %  all lower %
split source country                          
test  S1     France          0.00         0.00
             India           0.00         0.00
             US              0.00         0.00
      S2     France         21.05         6.40
             India          36.96        26.58
             US             20.54         5.89
      S3     France          5.66         6.74
             India          14.20        16.29
             US              3.30         6.24
train S1     India           0.00         0.00
             US              0.00         0.00
      S2     India          37.35        26.88
             US             21.84         6.73
      S3     India          14.16        16.70
             US              3.30         7.08

**Observation:** Source 1 names are never all-upper or all-lower (they use normal "Title Case"). Source 2 has many ALL-CAPS names,
and both S2/S3 contain all-lowercase names. Capitalization carries no meaning here,
so **lowercasing** will be one of our first cleaning steps.

## A3. Different writing scripts (Unicode)

Every character has a Unicode number. Each alphabet lives in its own number range, e.g.

| Script | Range | Example |
|---|---|---|
| Devanagari (Hindi, Marathi) | U+0900–U+097F | राम |
| Bengali | U+0980–U+09FF | রাম |
| Gurmukhi (Punjabi) | U+0A00–U+0A7F | ਰਾਮ |
| Gujarati | U+0A80–U+0AFF | રામ |
| Oriya | U+0B00–U+0B7F | ରାମ |
| Tamil | U+0B80–U+0BFF | ராம் |
| Telugu | U+0C00–U+0C7F | రామ |
| Kannada | U+0C80–U+0CFF | ರಾಮ |
| Malayalam | U+0D00–U+0D7F | രാമ |
| Accented Latin | U+00C0–U+024F | é, ñ, ç |

`str.contains("[\u0900-\u097F]")` is True if a text contains at least one Devanagari character.

In [6]:
scripts = {
    "Devanagari": "[\u0900-\u097F]",
    "Bengali": "[\u0980-\u09FF]",
    "Gurmukhi": "[\u0A00-\u0A7F]",
    "Gujarati": "[\u0A80-\u0AFF]",
    "Oriya": "[\u0B00-\u0B7F]",
    "Tamil": "[\u0B80-\u0BFF]",
    "Telugu": "[\u0C00-\u0C7F]",
    "Kannada": "[\u0C80-\u0CFF]",
    "Malayalam": "[\u0D00-\u0D7F]",
    "Accented Latin": "[\u00C0-\u024F]",
    "ANY non-English": "[^\x00-\x7F]",
}

name_scripts = pd.DataFrame({script: pct(names.str.contains(pattern)) for script, pattern in scripts.items()})
name_scripts.loc[:, (name_scripts > 0).any()]    # hide columns that are always 0

Devanagari  Bengali  Gurmukhi  Gujarati  Oriya  Tamil  Telugu  Kannada  Malayalam  Accented Latin  ANY non-English
split source country                                                                                                                    
test  S1     France         0.00     0.00      0.00      0.00   0.00   0.00    0.00     0.00       0.00           15.95            15.95
             India          0.00     0.00      0.00      0.00   0.00   0.00    0.00     0.00       0.00            0.00             0.00
             US             0.00     0.00      0.00      0.00   0.00   0.00    0.00     0.00       0.00            0.00             0.00
      S2     France         0.00     0.00      0.00      0.00   0.00   0.00    0.00     0.00       0.00           24.44            24.44
             India         13.43     1.51      0.35      1.47   0.39   1.64    1.93     2.03       0.96            3.97            27.69
             US             0.00     0.00      0.00      0.00   0.00   0.00    0.00     0.00       0.00            6.12             6.12
      S3     France         0.00     0.00      0.00      0.00   0.00   0.00    0.00     0.00       0.00           23.88            23.88
             India          7.58     0.86      0.18      0.82   0.20   0.95    1.17     1.09       0.56            4.90            18.30
             US             0.00     0.00      0.00      0.00   0.00   0.00    0.00     0.00       0.00            6.34             6.34
train S1     India          0.00     0.00      0.00      0.00   0.00   0.00    0.00     0.00       0.00            0.00             0.00
             US             0.00     0.00      0.00      0.00   0.00   0.00    0.00     0.00       0.00            0.00             0.00
      S2     India         13.30     1.58      0.32      1.49   0.38   1.60    1.90     1.81       0.93            4.35            27.66
             US             0.00     0.00      0.00      0.00   0.00   0.00    0.00     0.00       0.00            6.81             6.81
      S3     India          7.50     0.88      0.19      0.81   0.20   0.98    1.10     1.02       0.51            5.23            18.42
             US             0.00     0.00      0.00      0.00   0.00   0.00    0.00     0.00       0.00            6.86             6.86

In [7]:
indian_script = "[\u0900-\u0D7F]"
print("Indian-script names in train Source 2 / 3:")
sample.loc[names.str.contains(indian_script) & (sample["split"] == "train"),
           ["source", "business_name", "business_address"]].head(10)

Indian-script names in train Source 2 / 3:


,source,business_name,business_address
200008,S2,आनंद टेक प्रा. लि.,"S3 2ND FLOOR, HUZUR, BHOPAL, Madhya Pradesh"
200027,S2,सिल्वर इन्वेस्टमेंट्स प्राइवेट लिमिटेड,"#8/5 ADDHAR RESIDENCYMANGALWAR PETH, PUNE, Maharashtra"
200036,S2,శ్రీ మార్కెటింగ్ ప్రైవేట్ లిమిటెడ్,"2-131, KUMMRA VEEDHI KANDRAKOTA, GUNTUR, EAST GODAVARI, Andhra Pradesh"
200065,S2,फॉर्च्यून सिस्टम्स प्राइवेट लिमिटेड,"G-556 FF KH NO.56/12/1, WEST DELHI, Delhi"
200074,S2,హై కేర్ ప్రైవేట్ లిమిటెడ్,"D/396VAARTHALOWERTANKBUND ROAD, HYDERABAD, Telangana"
200077,S2,ರಿಯಲ್ ಇನ್ವೆಸ್ಟ್‌ಮೆಂಟ್ ಪ್ರೈವೇಟ್ ಲಿಮಿಟೆಡ್,"BANGALORE, Karnataka, DOOR NO 30/1A/A1"
200097,S2,सदर्न टेक फर्नीचर प्राइवेट लिमिटेड,"PLOT NO 31/2-B, F/F, KH NO-225 AND 1084, SHYAM VIHAR, PH-II, A-BLOCK NAJAFGARH, SOUTH ..."
200108,S2,डायनामिक प्रोडक्ट्स प्राइवेट लिमिटेड,"50 KA 2, JYOTI NAGAR, JAIPUR, राजस्थान"
200114,S2,बेस्ट लॉजिस्टिक्स प्राइवेट लिमिटेड,"PLOT 83 22-B15, PRATAP NAGAR, JAIPUR, Rajasthan"
200120,S2,ரெட் இம்பெக்ஸ் பிரைவேட் லிமிடெட்,"##13/94, TOWN MAIN ROAD, YERCAUD, SALEM, Tamil Nadu"


**Observations**
- **Source 1 never uses an Indian script.** Source 2/3 do for roughly 1 in 4 (S2) / 1 in 5 (S3) Indian names
  — Hindi, Tamil, Telugu, Kannada, Bengali, Gujarati, Punjabi, Malayalam, Oriya.
  An English name and a Hindi name have **zero characters in common**, so simple text similarity
  will score them 0. For these records the **address** has to carry the evidence (see part B).
- **Accents appear where they should not:** `Prívate`, `Wórks`, `Gríll`, `Córporation`, `Sôlidarite`.
  These are *noise* added to the data, on a random letter of a random word (real English words have no accents).
  French names also have real accents (`École`, `Établissements`).
  Either way, removing accents (`é → e`) is probably a safe cleaning step — to be tested in Phase 6.

In [8]:
accent_examples = sample.loc[names.str.contains("[\u00C0-\u024F]") & (sample["country"] != "France"), "business_name"]
accent_examples.head(10).tolist()

['hítechengineering.com',
 'TQE Facility Prívate Ltd - 4914786110',
 'Lucky Lucky Gríll',
 'Eco Infratech Có &',
 'Diamond Textile Wórks Co',
 'Blackburn Northwest Ínc',
 'Wadhwa Atm Prívate Limited',
 'Vium Córporation',
 '3 Grífols 30 East',
 'ZOO Products Cénter']

## A4. Punctuation and noise characters

Recipe: delete every letter, digit, mark and space with `[\p{L}\p{N}\p{M}\s]`; what is left are the symbols.
`.map(list)` splits each leftover string into single characters, and `explode()` puts each one on its own row so we can count them.

In [9]:
symbols_only = names.str.replace("[\\p{L}\\p{N}\\p{M}\\s]", "", regex=True)
punctuation = (
    symbols_only.map(list)
    .explode()
    .dropna()
    .groupby(sample["source"])
    .value_counts()
    .rename("count")
    .reset_index()
)
punctuation.pivot(index="business_name", columns="source", values="count").fillna(0).astype(int).sort_values("S2", ascending=False).head(15)

source,S1,S2,S3
business_name,,,
.,47646,79107,75278
",",30643,27900,27790
(,11483,22732,22948
),11486,22713,22944
-,3004,21587,22884
&,20203,16946,16729
],2,11151,11867
[,1,11150,11867
',6924,6319,6380


**Observations**
- S2/S3 add many brackets `[ ]`, `( )`, dashes, `*`, `#`, `>`, `|` that S1 almost never uses.
- One "symbol" in the list looks empty: it is **U+200C, an invisible "zero-width non-joiner"** used inside
  Indian-script words. Invisible characters make two strings look identical on screen but compare as different,
  so our cleaning must remove them too.

### Names that start with a symbol

Source 2/3 sometimes glue junk in front of a name: `***`, `>>`, `--`, `...`, `#`, `@`, `[LLC]`.

In [10]:
starts_with_symbol = names.str.match("^[^\\p{L}\\p{N}]")
print(pct(starts_with_symbol).unstack("country"))
sample.loc[starts_with_symbol & (sample["source"] != "S1"), "business_name"].head(15).tolist()

country       France  India    US
split source                     
test  S1        0.05   0.16  0.13
      S2        0.92   1.50  2.26
      S3        0.81   1.56  2.22
train S1         NaN   0.14  0.15
      S2         NaN   1.68  2.46
      S3         NaN   1.76  2.43


['#projectsdivine',
 '*** Borough Of South Corporation',
 '@mirellemoede',
 '@tannerpacific',
 '>> Prater Montage',
 '& MAXIM METOYER',
 '(Corp) 14/56 Andretti',
 '*** Chennai Basera  Private',
 '... Star Solutions  Automobiles',
 '... DELTA lNC. CENTER',
 '#cloverinternal',
 '[Corp] UKX Snlese Healthy',
 '... Enterprises Shiva Logistics Limited Partners',
 '(Co.) Modi Quotient',
 '>> 5ecure Racne LLC #79383']

### Bracketed words

S2/S3 wrap words in brackets: `Terri Taylor Peak LLC [Partners]`, `MS [Consultancy]`, `Davis Family (Office)`.
The word inside is usually a real part of the name, so later we should **remove the bracket characters, not the word**.

In [11]:
has_brackets = names.str.contains("[\\[\\]()]")
print(pct(has_brackets).unstack("country"))
sample.loc[has_brackets & (sample["source"] != "S1"), "business_name"].head(10).tolist()

country       France  India    US
split source                     
test  S1        7.95   5.25  0.00
      S2       10.24   9.44  6.28
      S3        9.63   9.96  6.44
train S1         NaN   5.15  0.00
      S2         NaN   9.59  6.52
      S3         NaN  10.19  6.47


['Terri Taylor Peak LLC [Partners]',
 'BRIGHT NAILS [RIVERSIDE]',
 'Phillips Graphics (Corp)',
 'Shree [Clinic]',
 'YR Metals Infratech Private (Limited)',
 'Buckley Haeh (PLLC)',
 'Marja Smith Piedmont Arcelormittal (LLC)',
 'DERMATOLOGY (CLINIC)',
 'Shiv Syntex (India) Private',
 'Marshall-First Nmp (Co)']

## A5. Website-style names

Some records use the business's web domain as its name: `maurewilliamscolombier.com`.
The words are glued together, so word-based comparison fails, but
**removing spaces** from the S1 name (`Maure Williams Colombier` → `maurewilliamscolombier`) makes them comparable.

In [12]:
domain_like = names.str.contains("\\.(?:com|net|org|in|co|fr)\\b", case=False) | names.str.contains("www\\.", case=False)
print(pct(domain_like).unstack("country"))
sample.loc[domain_like, "business_name"].head(10).tolist()

country       France  India    US
split source                     
test  S1        0.00   0.00  0.00
      S2        3.47   2.74  3.67
      S3        3.65   3.03  3.44
train S1         NaN   0.00  0.00
      S2         NaN   3.44  4.49
      S3         NaN   3.62  4.20


['hítechengineering.com',
 'FIRSTPACKAGINGDIGITAL.COM',
 'mejiastrategictalon.com',
 'rcmedical.com',
 'bidynepizzagrand.com',
 'gulfguild.com',
 'southernbaringsraleigh.com',
 'eliteapex.com',
 'krvproduct.com',
 'urologysafepartners.com']

### Hashtag / handle names

A few names look like social-media handles: `#safemarine`, `@tannerpacific`.

In [13]:
handle_like = names.str.match("^[#@][a-z0-9]+$", case=False)
print(pct(handle_like).unstack("country"))
sample.loc[handle_like, "business_name"].head(8).tolist()

country       France  India    US
split source                     
test  S1        0.00   0.00  0.00
      S2        0.55   0.35  0.50
      S3        0.51   0.37  0.49
train S1         NaN   0.00  0.00
      S2         NaN   0.45  0.63
      S3         NaN   0.49  0.64


['#projectsdivine',
 '@mirellemoede',
 '@tannerpacific',
 '#cloverinternal',
 '@GULFFELLOWSHIP',
 '@consultantsmagnificent',
 '#SURYAVENTURES',
 '@LIEX']

## A6. Most common words in names

Steps: lowercase → replace punctuation with spaces → split on spaces → `explode` → `value_counts`.
(`[^\p{L}\p{N}\p{M}\s]` = "anything that is not a letter, digit, mark or space", so accented and Indian-script words stay whole.)

In [14]:
def words(text_series):
    return (
        text_series.str.lower()
        .str.replace("[^\\p{L}\\p{N}\\p{M}\\s]", " ", regex=True)
        .str.split()
    )

name_words = words(names)

top_words = {}
for (source, country), group in name_words.groupby([sample["source"], sample["country"]]):
    top_words[f"{source} {country}"] = group.explode().value_counts().head(25).index.tolist()

pd.DataFrame(top_words)

,S1 France,S1 India,S1 US,S2 France,S2 India,S2 US,S3 France,S3 India,S3 US
0,sarl,limited,llc,sarl,limited,llc,sarl,limited,llc
1,sas,private,inc,sas,private,inc,sas,private,inc
2,club,ltd,and,france,ltd,l,france,ltd,partners
3,france,pvt,s,s,pvt,partners,club,pvt,center
4,de,india,c,club,लिमिटेड,center,s,india,l
5,eurl,llp,l,eurl,प्राइवेट,corp,de,लिमिटेड,corp
6,amicale,services,care,de,india,com,eurl,प्राइवेट,group
7,sa,solutions,of,sa,services,group,sa,services,com
8,maison,brothers,associates,amicale,llp,and,a,llp,and
9,du,trading,center,sasu,com,c,amicale,center,s


**Why are `l`, `c`, `s`, `p` so common in US names?** Punctuation was replaced by spaces, so
`L.L.C.` became `l l c`, `P.C.` became `p c`, and `Joe's` became `joe s`. In Phase 6 we should handle dotted
abbreviations by **deleting** dots inside them (`L.L.C.` → `llc`) rather than turning them into spaces.

**Observation:** the most frequent words are legal suffixes (`limited`, `private`, `llc`, `inc`, `sarl`)
and generic business words (`services`, `center`, `care`, `group`). Words that appear in thousands of names
say little about *which* business it is — this is exactly the idea behind **IDF** (Phase 8):
rare words (`colombier`, `mylapore`) are strong evidence, common words are weak evidence.

## A7. Legal suffixes

The legal form is usually the **last word** of the name. `.str[-1]` takes the last element of each word list.

In [15]:
last_word = name_words.str[-1]

top_last = {}
for (source, country), group in last_word.groupby([sample["source"], sample["country"]]):
    top_last[f"{source} {country}"] = [f"{w} ({n})" for w, n in group.value_counts().head(15).items()]

pd.DataFrame(top_last)

,S1 France,S1 India,S1 US,S2 France,S2 India,S2 US,S3 France,S3 India,S3 US
0,sarl (8512),limited (102601),llc (52602),sarl (4550),limited (37919),llc (26309),sarl (4508),limited (46108),llc (27736)
1,sas (5985),ltd (28541),inc (35502),sas (3283),ltd (23566),inc (20774),sas (3227),ltd (24635),inc (21039)
2,eurl (1975),llp (7679),c (6819),eurl (1379),लिमिटेड (16524),com (8114),eurl (1297),private (10088),com (7595)
3,sa (1466),co (3316),corp (4125),sa (1183),private (10073),corp (6959),sa (1108),लिमिटेड (8866),corp (6721)
4,sasu (1211),trust (1415),group (3981),sasu (1041),com (5289),center (6240),com (1041),com (5714),center (6324)
5,sci (943),corporation (1372),pc (3776),com (996),llp (4610),co (6004),sasu (1007),center (5215),partners (5560)
6,ei (497),company (1329),pllc (3020),sci (911),center (4089),ltd (5779),sci (901),llp (5200),co (5499)
7,club (318),corp (1313),center (2816),france (733),pvt (3162),partners (5564),france (747),services (4064),group (5197)
8,jean (197),group (1173),associates (2645),s (536),services (3140),group (5142),groupe (514),pvt (3051),ltd (5175)
9,france (190),clinic (991),lp (2442),groupe (534),लि (3111),c (4972),fils (475),co (2196),c (4594)


The same legal form is written in many ways. Let's count the variants per source:

In [16]:
suffix_variants = ["private", "prívate", "pvt", "pirtae",
                   "limited", "ltd", "límited",
                   "inc", "incorporated", "ínc",
                   "corp", "corporation", "córporation",
                   "co", "company", "cie",
                   "llc", "llp", "pllc",
                   "sarl", "sas", "sasu", "eurl", "sa", "sci", "snc"]

all_name_words = name_words.explode()
found = all_name_words[all_name_words.isin(suffix_variants)]
pd.crosstab(found.values, sample.loc[found.index, "source"].values, rownames=["written as"], colnames=["source"]).reindex(suffix_variants).fillna(0).astype(int)

source,S1,S2,S3
written as,,,
private,85000,46037,53549
prívate,0,1083,1314
pvt,23758,16620,18268
pirtae,0,0,3
limited,102694,47191,57290
ltd,28937,33477,34346
límited,0,1230,1544
inc,35502,26273,26352
incorporated,2,1590,1358


### Does the suffix stay at the end?

In Source 2/3 the suffix sometimes moves to the **front** (`LLC Moncada Learning Center`, `Pvt XRW Moulding Ltd`).

In [17]:
first_word = name_words.str[0]
legal_words = {"llc", "inc", "corp", "ltd", "pvt", "private", "limited", "sarl", "sas", "co"}
print(pct(first_word.isin(legal_words)).unstack("country"))
sample.loc[first_word.isin(legal_words) & (sample["source"] != "S1"), "business_name"].head(10).tolist()

country       France  India    US
split source                     
test  S1        0.04   0.02  0.01
      S2        3.35   3.96  3.30
      S3        3.16   4.47  3.05
train S1         NaN   0.01  0.01
      S2         NaN   3.86  3.00
      S3         NaN   4.40  2.77


['Co Mckenzie Be Colombier',
 'INC. THAYER TAXABLE',
 'Ltd Patrick & Davis',
 "LLC Deena's Alfics",
 'inc jacobo royal care',
 'Ltd Oncology Clinic Associates',
 "Inc Fletcher's Massage",
 'Private Dial Limited (Appliances)',
 'LLC Total Health Studios',
 'Pvt Sneha Analyiacs Ltd']

**Observations on legal suffixes**
- The same form appears as `Private`/`Pvt`, `Limited`/`Ltd`, `Inc`/`Incorporated`, `Corp`/`Corporation`.
- Suffixes can move to the front, and appear in Hindi (`प्राइवेट लिमिटेड` = "private limited").
- Suffixes are **very common**, so they are weak evidence: `ABC Pvt Ltd` vs `XYZ Pvt Ltd` share 2 of 4 words
  but are different businesses. In Phase 6 we will test two options: map variants to one form
  (`pvt → private`) or remove suffixes from the name before comparing — and keep whichever scores better.

## A8. Duplicate names

Many different records share a name, even inside Source 1 (which is deduplicated!). So two records with an identical
name are **not automatically** the same business — we need the address too.

In [18]:
s1_names = sample.loc[(sample["split"] == "train") & (sample["source"] == "S1"), ["business_name", "business_address"]]
repeated = s1_names["business_name"].value_counts()
print("names that appear more than once in the S1 sample:", (repeated > 1).sum())
top_repeated = repeated.index[0]
print("example:", top_repeated)
s1_names[s1_names["business_name"] == top_repeated].head(8)

names that appear more than once in the S1 sample: 12414
example: Family Group


,business_name,business_address
22131,Family Group,"IL, Glenview, 129 Julie Drive"
32596,Family Group,"4510 Wellborn Woods Drive, Eight Mile, AL"
32712,Family Group,"201 Private Road 5275, Buna, TX"
37121,Family Group,"12 Oak Hill Lane, St Peters, MO"
40128,Family Group,"906 Union Avenue, Wheaton, IL"
43755,Family Group,"3119 London Avenue, Brookhaven, NY"
47045,Family Group,"11433 Jordan Farms Road, South Jordan, UT"
51071,Family Group,"75 White Street, Manchester, CT"


---
# Part B — Addresses

## B1. Empty addresses and placeholder values

Besides truly empty addresses, S2/S3 contain the **text** `null` or `<NULL>` inside addresses
(`"5 MARY ELLA COURT, null, SMRYNA, DE"`). These placeholders carry no information and should be removed.

In [19]:
pd.DataFrame({
    "empty %": pct(addresses == ""),
    "contains null/<NULL> %": pct(addresses.str.contains("\\bnull\\b", case=False)),
}).unstack("country")

empty %             contains null/<NULL> %            
country       France India    US                 France India    US
split source                                                       
test  S1        0.00  0.00  0.00                    0.0  0.00  0.00
      S2        3.20  2.28  2.99                    0.0  2.13  3.02
      S3        2.98  2.39  2.76                    0.0  2.18  2.88
train S1         NaN  0.00  0.00                    NaN  0.01  0.00
      S2         NaN  2.88  3.65                    NaN  2.15  2.86
      S3         NaN  3.12  3.51                    NaN  2.02  2.70

In [20]:
sample.loc[addresses.str.contains("\\bnull\\b", case=False), ["source", "business_address"]].head(8)

,source,business_address
39798,S1,"176/1, Second Floor, Hari Nagar Ashram, Null, New Delhi, South Delhi, Delhi"
40822,S1,"Hyderabad, 30-148/27, Tirumalagiri, Telangana, Plot No 28 Old, Safilguda Null"
86949,S1,"25, Park Lane, Null, Kolkata, Kolkata, Howrah, West Bengal"
103257,S1,"12Th Cross 6Th Main Gopalappa Building Devasandra Kr Puram Null, Bangalore, Karnataka"
118867,S1,"14 T P, Venugopal L/O, Null, Bangalore, Karnataka"
126614,S1,"C/O Madhabilata Rout, Null Siriapur, Town-I, Jajpur, Orissa"
127063,S1,"371, Yar Mohd.Plot, Char Null, Jawahar Nagar, Santacruz(E), Mumbai, Mumbai City, Mahar..."
151816,S1,"Lig-49, Aishbaag Pill Col, Old Labour Colony Null, Lucknow, Uttar Pradesh"


## B2. Length and number of parts

Addresses are made of comma-separated parts (street, area, city, state).

In [21]:
sample.assign(
    address_length=addresses.str.len(),
    comma_parts=addresses.str.count(",") + 1,
).groupby(["source", "country"])[["address_length", "comma_parts"]].median()

address_length  comma_parts
source country                             
S1     France             49.0          3.0
       India              76.0          6.0
       US                 34.0          3.0
S2     France             40.0          3.0
       India              68.0          5.0
       US                 32.0          3.0
S3     France             41.0          3.0
       India              59.0          4.0
       US                 39.0          3.0

**Observation:** Indian addresses have more parts (building, floor, street, area, landmark, city, state)
than US/French ones (street, city, state).

## B3. Abbreviations: `Street` vs `St`, `Road` vs `Rd` ...

We count how often each form appears as a separate word, per source.

In [22]:
address_words = words(addresses).explode()
group_label = sample["source"] + " " + sample["country"]

words_to_check = ["street", "st", "road", "rd", "drive", "dr", "avenue", "ave", "lane", "ln",
                  "boulevard", "blvd", "apartment", "apt", "rue", "r", "bd", "av"]
found = address_words[address_words.isin(words_to_check)]

# crosstab = count every (word, group) combination
abbreviation_table = pd.crosstab(found.values, group_label.loc[found.index].values, rownames=["word"], colnames=["group"]).reindex(words_to_check).fillna(0).astype(int)
abbreviation_table[[c for c in abbreviation_table.columns if "India" not in c]]

group,S1 France,S1 US,S2 France,S2 US,S3 France,S3 US
word,,,,,,
street,0,40623,0,17375,0,18567
st,12,1459,480,20854,468,20011
road,0,36903,0,16287,0,17006
rd,12,606,5,19481,10,18924
drive,0,32734,0,14035,0,14874
dr,2,421,0,17594,3,16928
avenue,3544,26806,1906,11625,1861,12217
ave,0,442,460,14555,414,13864
lane,0,13944,0,5902,0,6464


**Observation:** Source 1 (US) writes `Street`, `Road`, `Drive`; Source 2/3 often abbreviate to `St`, `Rd`, `Dr`.
French S2 abbreviates `Rue → R.`, `Boulevard → BD`, `Avenue → AV`.
If we compare `street` vs `st` as words, they don't match — mapping abbreviations to one form may help (to be measured).
Careful: `dr` also means **Doctor** in names, and `st` can mean **Saint** — so abbreviation rules for addresses
should only be applied to addresses.

## B4. State / region format

The **last** comma-separated part of an address is usually the state (US, India) or region (France).

In [23]:
last_part = addresses.str.split(",").str[-1].str.strip()

top_last_part = {}
for (source, country), group in last_part.groupby([sample["source"], sample["country"]]):
    top_last_part[f"{source} {country}"] = group.value_counts().head(12).index.tolist()
pd.DataFrame(top_last_part)

,S1 France,S1 India,S1 US,S2 France,S2 India,S2 US,S3 France,S3 India,S3 US
0,Hauts-de-France,Maharashtra,TX,Hauts-de-France,Maharashtra,TX,Hauts-de-France,MH,Texas
1,Nouvelle-Aquitaine,Delhi,NY,Nouvelle-Aquitaine,Delhi,NY,Nouvelle-Aquitaine,DL,New York
2,Pays de la Loire,Uttar Pradesh,NC,Gironde,Uttar Pradesh,NC,Nord,UP,North Carolina
3,Bordeaux,Karnataka,OH,Nord,Karnataka,OH,Pays de la Loire,KA,Ohio
4,Nantes,Tamil Nadu,IL,Pays de la Loire,महाराष्ट्र,IL,Gironde,महाराष्ट्र,Illinois
5,Lille,Gujarat,VA,Loire-Atlantique,Tamil Nadu,VA,Loire-Atlantique,TN,Tennessee
6,Tourcoing,West Bengal,TN,BORDEAUX,West Bengal,TN,Bordeaux,WB,Virginia
7,Dunkerque,Telangana,MA,NANTES,Gujarat,MA,Nantes,GJ,Arizona
8,Calais,Haryana,AZ,LILLE,Telangana,AZ,Lille,TG,Massachusetts
9,Roubaix,Kerala,IN,,दिल्ली,,,,


**Observations — the same state is written in different forms by different sources:**

| Country | Source 1 | Source 2 | Source 3 |
|---|---|---|---|
| US | `TX` (code) | `TX` (code) | `Texas` (full name) |
| India | `Maharashtra` (full name) | `Maharashtra` or `महाराष्ट्र` (Hindi) | `MH` (code), or native script |
| France (test) | `Hauts-de-France` (region) | `Nord` / `Gironde` (department) or region | similar |

So a US S1 record says `TX` and its S3 match says `Texas`. Word overlap sees two different words.
A small **state-name ↔ code** lookup learned *from the training data itself* could fix this — we'll decide in Phase 6.
French regions ↔ departments can't be learned from train (France is test-only), so our method must not depend on it.

## B5. Numbers and postal codes

House numbers are very useful evidence: `85 Wayne Avenue` vs `85 Wanye Avenue` share the number `85` exactly,
even though the street has a typo. Let's see how often addresses contain numbers, and whether postal codes exist
(US ZIP = 5 digits, India PIN = 6 digits, France = 5 digits).

`(?<!\d)\d{6}(?!\d)` means "exactly 6 digits, not part of a longer number".

In [24]:
pd.DataFrame({
    "has any digit %": pct(addresses.str.contains("\\d")),
    "starts with a number %": pct(addresses.str.match("^\\d")),
    "5-digit number %": pct(addresses.str.contains("(?<!\\d)\\d{5}(?!\\d)")),
    "6-digit number %": pct(addresses.str.contains("(?<!\\d)\\d{6}(?!\\d)")),
}).unstack("country")

has any digit %                starts with a number %               5-digit number %              6-digit number %            
country               France  India      US                 France  India     US           France India     US           France India    US
split source                                                                                                                               
test  S1               99.56  91.21  100.00                  86.06  25.36  85.82             0.38  0.30  10.78             0.01  0.01  0.15
      S2               92.95  93.11   91.97                  66.90  18.34  74.30             0.60  1.07  10.90             0.00  0.01  1.44
      S3               93.31  91.87   93.00                  67.55  18.14  74.71             0.51  0.97  10.88             0.02  0.03  1.41
train S1                 NaN  91.40  100.00                    NaN  25.92  86.10              NaN  0.27  10.98              NaN  0.02  0.12
      S2                 NaN  91.31   90.13                    NaN  18.08  72.95              NaN  1.08  10.53              NaN  0.03  1.38
      S3                 NaN  90.07   91.29                    NaN  18.00  73.32              NaN  1.07  10.61              NaN  0.02  1.29

In [25]:
print("Examples of 5-digit numbers in US addresses - these are house numbers, not ZIP codes:")
sample.loc[addresses.str.contains("(?<!\\d)\\d{5}(?!\\d)") & (sample["country"] == "US"), "business_address"].head(6).tolist()

Examples of 5-digit numbers in US addresses - these are house numbers, not ZIP codes:


['28486 Lamar Avenue, Chisago City, MN',
 'Unit Apartment 103, WI, Village Of Sussex, N59W23320 Clover Drive',
 '2000 Arapaho Road, Unit 21316, Richardson, TX',
 '17136 136th Place, Renton, WA',
 'WI, City Of Brookfield, 19125 Thomson Drive, Unit Unit 202',
 'Town Of Norway, 22428 Hanson Road, WI']

**Observation:** there are **basically no postal codes** in this data (6-digit PINs ≈ 0%; 5-digit numbers in the US
are house numbers like `28486 Lamar Avenue`). So we cannot rely on postal codes; **house / plot / door numbers** are the
useful numeric evidence. Numbers must be **kept** during cleaning.

## B6. Landmark-based addresses

Indian addresses often describe location relative to a landmark: `Opp Ratna Hospital`, `Near Govt School`.

In [26]:
landmark = addresses.str.contains("\\b(?:opp|opposite|near|nr|behind|beside|next to)\\b", case=False)
print(pct(landmark).unstack("country"))
sample.loc[landmark, "business_address"].head(6).tolist()

country       France  India    US
split source                     
test  S1        0.00  13.41  0.01
      S2        0.01  11.49  0.01
      S3        0.00   8.64  0.01
train S1         NaN  13.48  0.00
      S2         NaN  11.64  0.01
      S3         NaN   8.78  0.01


['Juni Gadhi, Gujarat, Vadodara, Near Bhadra Kacheri Near Dist. Police Station, Panigate',
 'Rajender Kumar, Nizampur, Road, Opp Shanti Hospital, Narnaul, Mahendragarh, Haryana',
 '16, 3Rd Floor, Plot No. 80, Kumar Niwas Chs Pitambar Lane, Near Mahim Railway Statio, N, Mahim, Mumbai City, Maharashtra',
 'Plot No.286, Opp.Register Office Sai Nagar, Sabbavaram, Visakhapatnam, Vishakhapatnam, Andhra Pradesh',
 'C/O Sambhu Dey Binnaguri, Near Shiv Mandir, Mal, Jalpaiguri, West Bengal',
 'P141, 14Th Floor, Tower P, Near Haldiram Dlf Capital Greens Phase 3, Moti Nagar, Delhi, West Delhi, Delhi']

**Observation:** ~9–13% of Indian addresses contain a landmark; US and France essentially never do.
Landmark words (`near`, `opp`) themselves are common filler, but the landmark name (`Ratna Hospital`) is useful evidence.

## B7. Word order

The same address parts can appear in a different order: `Lebanon, Tennessee, 88 Olive Cir` vs `88 Olive Circle, Lebanon, TN`.
A simple character-by-character comparison would punish this; **word-set** comparisons (Phase 8) ignore order.

In [27]:
us_addresses = sample.loc[sample["country"] == "US"]
print(pct(addresses.str.match("^\\d")).unstack("country")["US"].rename("US: starts with house number %"))
us_addresses.loc[~us_addresses["business_address"].str.match("^\\d") & (us_addresses["business_address"] != ""),
                 ["source", "business_address"]].head(8)

split  source
test   S1        85.82
       S2        74.30
       S3        74.71
train  S1        86.10
       S2        72.95
       S3        73.32
Name: US: starts with house number %, dtype: float64


,source,business_address
7,S1,"Unit Apartment 103, WI, Village Of Sussex, N59W23320 Clover Drive"
29,S1,"WI, City Of Brookfield, 19125 Thomson Drive, Unit Unit 202"
33,S1,"Town Of Norway, 22428 Hanson Road, WI"
42,S1,"Cobleskill, 105 Rensselaer Avenue, NY"
59,S1,"Jacksonville, AL, Rm 3106, 115 College Street"
68,S1,"IL, 459 Berkley Avenue, Elmhurst"
86,S1,"Frisco, 15655 Buffalo Cree Drive, TX"
97,S1,"AL, Troy, Unit Lot 2, 744 6611"


## B8. Indian-script addresses

Addresses also appear in Indian scripts, but usually only **partly** — mostly the state name
(`..., Chennai, தமிழ்நாடு`). The street part stays in English, which is good news for matching.

In [28]:
address_script = pct(addresses.str.contains(indian_script))
fully_indian = pct(addresses.str.contains(indian_script) & ~addresses.str.contains("[A-Za-z]"))
pd.DataFrame({"has Indian script %": address_script, "ONLY Indian script (no English letters) %": fully_indian}).unstack("country")

has Indian script %             ONLY Indian script (no English letters) %           
country                   France  India   US                                    France India   US
split source                                                                                     
test  S1                     0.0   0.00  0.0                                       0.0   0.0  0.0
      S2                     0.0  23.87  0.0                                       0.0   0.0  0.0
      S3                     0.0  23.14  0.0                                       0.0   0.0  0.0
train S1                     NaN   0.00  0.0                                       NaN   0.0  0.0
      S2                     NaN  23.39  0.0                                       NaN   0.0  0.0
      S3                     NaN  22.36  0.0                                       NaN   0.0  0.0

---
# Part C — Country

## C1. Country values per file (full files, from notebook 01)

| File | France | India | US |
|---|---|---|---|
| train S1 / S2 / S3 | 0 | 0.88M / 2.02M / 2.12M | 1.32M / 3.02M / 3.17M |
| test S1 / S2 / S3 | 0.26M / 0.70M / 0.73M | 0.81M / 2.31M / 2.41M | 0.66M / 1.87M / 1.95M |

Country is always written the same way (`US`, `India`, `France`) — no `USA` / `United States` variants, no empty values.

In [29]:
sample.groupby(["split", "source"])["country"].value_counts().unstack().fillna(0).astype(int)

country       France  India      US
split source                       
test  S1       29823  93550   76627
      S2       28695  94808   76497
      S3       28548  94808   76644
train S1           0  80046  119954
      S2           0  79959  120041
      S3           0  80073  119927

## C2. Do matching records ever have different countries?

This uses the **full** ground truth. We only need `entity_id` and `country` columns (`usecols=`),
which keeps memory small.

In [30]:
gt = pd.read_csv(TRAIN_DIR + "train_ground_truth.tsv", sep="\t", keep_default_na=False)
true_pairs = gt[gt["matched_entity_ids"] != ""].copy()
true_pairs["match_id"] = true_pairs["matched_entity_ids"].str.split(",")
true_pairs = true_pairs.explode("match_id")[["source1_entity_id", "match_id"]]
del gt

s1_country = pd.read_csv(TRAIN_DIR + "train_source1.tsv", sep="\t", keep_default_na=False, usecols=["entity_id", "country"])
true_pairs = true_pairs.merge(s1_country.rename(columns={"entity_id": "source1_entity_id", "country": "s1_country"}), on="source1_entity_id")
del s1_country

other_country = pd.concat([
    pd.read_csv(TRAIN_DIR + "train_source2.tsv", sep="\t", keep_default_na=False, usecols=["entity_id", "country"]),
    pd.read_csv(TRAIN_DIR + "train_source3.tsv", sep="\t", keep_default_na=False, usecols=["entity_id", "country"]),
])
true_pairs = true_pairs.merge(other_country.rename(columns={"entity_id": "match_id", "country": "match_country"}), on="match_id")
del other_country
gc.collect()

print("true pairs checked:", f"{len(true_pairs):,}")
pd.crosstab(true_pairs["s1_country"], true_pairs["match_country"])

true pairs checked: 7,638,365


match_country,India,US
s1_country,,
India,3059843,0
US,0,4578522


**Observation:** every one of the 7.6 million true pairs has the **same country** on both sides.
So we only ever need to compare an S1 record with S2/S3 records **of the same country**.
This is our first, safe blocking rule — it works for France too, because we never list the countries by hand:
we simply group by whatever country values appear.

In [31]:
del true_pairs
gc.collect();

---
# Summary — what EDA tells us to do next

| What we found | Where | What it suggests (to be **tested**, not assumed) |
|---|---|---|
| Random UPPER / lower case | S2, S3 | lowercase everything |
| Fake accents (`límited`, `ínc`) + real French accents | S2, S3, France | remove accents (`é → e`) |
| Junk prefixes `***`, `>>`, `--`, `#`, `@` and brackets `[ ]` `( )` | S2, S3 | strip symbols, keep the words inside brackets |
| `null` / `<NULL>` inside addresses | S2, S3 | remove placeholder words |
| Legal suffix variants (`Pvt`/`Private`, `Ltd`/`Limited`, `Inc`, `Corp`), sometimes moved to the front | all | map variants to one form **or** drop them — compare both |
| Website names (`maurewilliamscolombier.com`) | S2, S3 (~3–4%) | also compare names with spaces removed |
| Names in 9 Indian scripts | S2, S3 India (~20–28%) | rely on address for these; transliteration is an optional later step |
| `Street`↔`St`, `Road`↔`Rd`, `Rue`↔`R.` | S2, S3 | normalize address abbreviations (addresses only) |
| State as code vs full name vs native script | S3 (US full names, India codes) | state lookup learned from train, or just ignore the state part |
| House numbers present, **postal codes absent** | all | keep numbers; add a "shared numbers" feature |
| Address parts reordered | S2, S3 | use word-set similarity, not only character order |
| Landmarks (`Near`, `Opp`) | India (~10%) | treat `near`/`opp` as filler words |
| Same name ≠ same business | all | never match on name alone |
| True matches are always the same country | all | block by country (open-ended, works for France) |